# Evaluate the fine-tuned adapter — Column Description

Runs **both** the untuned base model (baseline) **and** the SFT+DPO fine-tuned adapter on the **held-out test examples** (read from `sft_data/test_examples.jsonl`, written fully-rendered by `finetune_descriptions.ipynb` — so eval prompts are byte-identical to training prompts by construction), then scores each generated description against the gold column description and shows a **side-by-side comparison**:

- **ROUGE-1/2/L** — lexical overlap (CPU, no download)
- **BERTScore** — semantic similarity (downloads an embedding model)
- **Length / verbosity ratio** — generated tokens ÷ reference tokens, flagged above the proposal's brevity threshold, plus the share of generations that hit the max-token ceiling
- **Deterministic WA plain-language checks** — word-count window, sentence length (mean words/sentence and share of sentences over the WA 20-word rule — the failure mode observed in base-model testing), unexpanded acronyms, Flesch-Kincaid grade, WA jargon list, "deadly 7 verbs" — mirroring the capstone Output Evaluation Report (§9 / §14.6); free, reproducible, and they measure exactly what the fine-tune claims to improve

**How to read the results.** Test targets are *raw* human text while training targets were curated, so with noisy golds the WA-check deltas (especially sentence length, word window, jargon) and the length ratio are the primary outcome — judged against the gold-reference calibration row, which shows what sponsor-quality human text scores on the same rules. Success looks like the tuned model **beating the human golds on the plain-language checks** while staying grounded; ROUGE/BERTScore serve as grounding sanity checks (they catch off-topic output, especially on the novel-target subset) rather than the headline.

A **gold benchmark** section re-scores the same predictions on just the sponsor "golden" datasets (held out of training), and a **novel-target subset** re-scores only the test examples whose gold text does **not** also appear among the train targets (a boilerplate-leakage control). An **improve-path slice** re-runs generation with a degraded gold in the existing-description REFERENCE block — production's *improve* flow — and reports an **echo rate** (how often a variant just copies the degraded reference).

**Both variants run by default** — there is no toggle. The base model is loaded **once**; the adapter is switched on for the fine-tuned pass and switched off (`model.disable_adapter()`) for the baseline pass, so no second model load is needed.

**Where this runs:** generation needs the GPU; the metric math is CPU-light.

## 0. Install (GPU host only)

In [ ]:
%pip install -q --upgrade "transformers>=5.11" "peft>=0.13" "bitsandbytes>=0.45" "datasets>=2.20" "accelerate>=0.34" "rouge-score>=0.1.2" "bert-score>=0.3.13" pandas
# transformers>=5.11 = the gemma4 architecture + the PR #45454 text-only fix.

## 1. Configuration

In [ ]:
from pathlib import Path

# ── Evaluation mode ────────────────────────────────────────────────────────
# This notebook evaluates BOTH variants in a single run and compares them:
#   • BASELINE   — the raw untuned Gemma-4-31B (adapter disabled)
#   • FINE-TUNED — the SFT+DPO adapter (adapter enabled)
# No toggle needed. The base model is loaded once; the adapter is flipped
# on/off in-place between the two generation passes.

MODEL_ID = "google/gemma-4-31B"

METADATA_PATH = Path("all_metadata.json")  # golden flags for the gold benchmark
TEST_EXAMPLES_PATH = Path(
    "sft_data/test_examples.jsonl"
)  # written by finetune_descriptions.ipynb
ADAPTER_PATH = "adapters/gemma-4-31b-coldesc-dpo"

# Per-variant + comparison output paths.
BASELINE_PRED_PATH = Path("baseline_predictions.jsonl")
BASELINE_RESULTS_PATH = Path("baseline_results.json")
FINETUNED_PRED_PATH = Path("eval_predictions.jsonl")
FINETUNED_RESULTS_PATH = Path("eval_results.json")
COMPARISON_PATH = Path("comparison_results.json")

# generation — column descriptions target ~50 words; 150 max tokens matches the
# production tool's column default (~2x the target, per the Model Tuning Report).
# The old 96 guaranteed the verbose baseline always truncated mid-sentence.
MAX_NEW_TOKENS = {"column_description": 150}

# metric toggles
DO_ROUGE, DO_BERTSCORE, DO_LENGTH, DO_WA_CHECKS = True, True, True, True
BERTSCORE_MODEL = "roberta-large"  # swap for a smaller model if bandwidth is tight
VERBOSITY_THRESHOLD = 1.15  # flag gen/ref length ratio above this

# limit examples while smoke-testing (None = all test examples)
MAX_EVAL_PER_TASK = None

print(f"Model    : {MODEL_ID}")
print(f"Adapter  : {ADAPTER_PATH}")
print(f"Variants : BASELINE (untuned) + FINE-TUNED (SFT+DPO)")

In [ ]:
# --- Run environment: Colab (Drive) OR local / remote GPU server ---
# Off Colab (your own GPU box) everything stays on the local disk; export PROJECT_DIR
# to choose where artifacts + the HF cache live (defaults to "."). Needs all_metadata.json
# + sft_data/test_examples.jsonl from the finetune step.
from pathlib import Path
import os

try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = IN_COLAB  # set False to use Colab's local /content disk instead of Drive
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path(
        os.environ.get("PROJECT_DIR", "/content/drive/MyDrive/MetadataFineTuningTool")
    )  # same Drive folder used for training
else:
    PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "."))
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Hugging Face auth: gated bases need a token off Colab. Read HF_TOKEN
# from the environment, or from a .env file (KEY=VALUE) in the cwd or PROJECT_DIR.
if not os.environ.get("HF_TOKEN"):
    for _env in (Path(".env"), PROJECT_DIR / ".env"):
        if _env.exists():
            for _line in _env.read_text().splitlines():
                if _line.strip().startswith("HF_TOKEN="):
                    os.environ["HF_TOKEN"] = _line.split("=", 1)[1].strip().strip("\"'")
            break
print(
    "HF token:", "set" if os.environ.get("HF_TOKEN") else "not set (public models only)"
)

# Re-root all paths under PROJECT_DIR.
CACHE_DIR = PROJECT_DIR / "hf_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
METADATA_PATH = PROJECT_DIR / "all_metadata.json"
TEST_EXAMPLES_PATH = PROJECT_DIR / "sft_data" / "test_examples.jsonl"
ADAPTER_PATH = str(PROJECT_DIR / "adapters" / "gemma-4-31b-coldesc-dpo")
BASELINE_PRED_PATH = PROJECT_DIR / "baseline_predictions.jsonl"
BASELINE_RESULTS_PATH = PROJECT_DIR / "baseline_results.json"
FINETUNED_PRED_PATH = PROJECT_DIR / "eval_predictions.jsonl"
FINETUNED_RESULTS_PATH = PROJECT_DIR / "eval_results.json"
COMPARISON_PATH = PROJECT_DIR / "comparison_results.json"

# eval needs the metadata (golden flags) and the rendered test examples
if (not METADATA_PATH.exists() or not TEST_EXAMPLES_PATH.exists()) and IN_COLAB:
    print("Upload all_metadata.json and/or test_examples.jsonl:")
    from google.colab import files

    up = files.upload()
    if "all_metadata.json" in up:
        METADATA_PATH.write_bytes(up["all_metadata.json"])
    if "test_examples.jsonl" in up:
        TEST_EXAMPLES_PATH.parent.mkdir(parents=True, exist_ok=True)
        TEST_EXAMPLES_PATH.write_bytes(up["test_examples.jsonl"])

assert METADATA_PATH.exists() and TEST_EXAMPLES_PATH.exists(), (
    f"Need all_metadata.json + sft_data/test_examples.jsonl under {PROJECT_DIR} "
    "(run finetune_descriptions.ipynb first, or set PROJECT_DIR)."
)

print(f"Adapter  : {ADAPTER_PATH}")
print(f"Variants : BASELINE (untuned) + FINE-TUNED (SFT+DPO)")

## 2. Load the held-out test examples

All prompt building lives in `finetune_descriptions.ipynb`, which loads the production prompts and writes the **fully-rendered** test examples to `sft_data/test_examples.jsonl`. This notebook only reads that file, so eval prompts are byte-identical to the prompts the adapter trained against — no duplicated builder code to drift.

In [ ]:
import json

# Fully-rendered prompts (system + user turns) and gold targets, exactly as the
# adapter saw them in training. Each row: uid, task, column, prompt_messages,
# messages, target, target_in_train (boilerplate-overlap flag for section 10),
# and — when the gold could be degraded — improve_prompt_messages /
# improve_existing for the improve-path slice (section 11).
eval_examples = [
    json.loads(line)
    for line in TEST_EXAMPLES_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print(f"Loaded {len(eval_examples)} test examples from {TEST_EXAMPLES_PATH}")

## 3. Load metadata (golden flags) + finalize the eval set

In [ ]:
records = json.loads(METADATA_PATH.read_text(encoding="utf-8"))  # golden flags

if MAX_EVAL_PER_TASK:
    eval_examples = eval_examples[:MAX_EVAL_PER_TASK]

n_seen = sum(1 for e in eval_examples if e.get("target_in_train"))
print(
    f"test datasets: {len({e['uid'] for e in eval_examples})} | "
    f"column_description examples: {len(eval_examples)}"
)
print(
    f"novel targets: {len(eval_examples) - n_seen} | "
    f"boilerplate overlap with train: {n_seen}"
)

## 4. Load base + adapter *(GPU)*

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# gemma4 is a native transformers architecture (v5.11+): no trust_remote_code.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
# google/gemma-4-31B is a BASE checkpoint with no chat template; install the
# same plain-text template training used. The template OWNS the special tokens:
# "{{ bos_token }}" once at the start and eos_token after each ASSISTANT turn
# (that EOS is what lets the fine-tuned model stop on its own). This string must
# be BYTE-IDENTICAL to the one in finetune_descriptions.ipynb — in particular
# "ASSISTANT:" has NO trailing space (the completion supplies the leading
# space). A one-character difference would make every eval prompt diverge from
# the training prompts; scripts/check_consistency.py asserts the two copies match.
if tokenizer.chat_template is None:
    tokenizer.chat_template = "{{ bos_token }}{% for message in messages %}{{message['role'].upper() + ': ' + message['content'] + (eos_token if message['role'] == 'assistant' else '') + '\n\n'}}{% endfor %}{% if add_generation_prompt %}ASSISTANT:{% endif %}"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",  # the gemma4 docs' recommended implementation
    cache_dir=CACHE_DIR,
)


# Load the adapter once. Both passes share this single model: the adapter is
# ENABLED for the fine-tuned pass and DISABLED for the baseline pass.
adapter_exists = bool(ADAPTER_PATH) and Path(ADAPTER_PATH).exists()
if adapter_exists:
    from peft import PeftModel

    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    has_adapter = hasattr(model, "disable_adapter")
    print("Loaded adapter:", ADAPTER_PATH)
else:
    has_adapter = False
    print(
        f"⚠ No adapter found at {ADAPTER_PATH} — only the BASELINE "
        "will be evaluated (the fine-tuned pass + comparison will be skipped)."
    )
model.eval()

## 5. Generate predictions for both variants *(GPU)*

Runs the test set twice through the **same** loaded model: once with the adapter **disabled** (baseline) and once with it **enabled** (fine-tuned). Both prediction sets are kept in memory (`baseline_predictions`, `finetuned_predictions`) in the same order as `eval_examples`, and each is also written to its own `.jsonl`.

In [ ]:
import json, time, contextlib


def generate(prompt_messages, max_new_tokens):
    text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    # The chat template owns BOS — add_special_tokens=False keeps eval
    # tokenization identical to how TRL tokenized the training text (single BOS).
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(
        model.device
    )
    with torch.no_grad():
        # The FINE-TUNED model stops at <eos> (trained in via the template). The
        # UNTUNED baseline never learned to stop, so stop_strings halts it at the
        # template's blank-line turn separator instead of burning the full token
        # budget — which also makes hit_max measure real ceiling collisions
        # rather than "the baseline always rambles".
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            stop_strings=["\n\n"],
            tokenizer=tokenizer,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1] :]
    hit_max = new_tokens.shape[0] >= max_new_tokens  # ran into the token ceiling
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)
    # Belt-and-suspenders: stop_strings leaves the stop text in the output, so
    # still cut at the first fake follow-on turn. (Gold targets contain no
    # newlines, so this never clips a legitimate completion.)
    for stop in ("\n\n", "\nUSER:", "\nSYSTEM:", "\nASSISTANT:"):
        decoded = decoded.split(stop)[0]
    return decoded.strip(), hit_max


def run_variant(label, adapter_enabled):
    """Generate predictions for every eval example under one model configuration.

    adapter_enabled=False runs inside `model.disable_adapter()` (the baseline);
    adapter_enabled=True runs the model as-loaded (the fine-tuned adapter).
    """
    if adapter_enabled or not has_adapter:
        ctx = contextlib.nullcontext()
    else:
        ctx = model.disable_adapter()  # turn the adapter OFF for the baseline pass

    preds = []
    start_time = time.time()
    print(f"\n=== Generating: {label} ===")
    with ctx:
        for i, ex in enumerate(eval_examples, 1):
            pred, hit_max = generate(ex["prompt_messages"], MAX_NEW_TOKENS[ex["task"]])
            preds.append(
                {
                    "uid": ex["uid"],
                    "task": ex["task"],
                    "column": ex["column"],
                    "prediction": pred,
                    "reference": ex["target"],
                    "hit_max": hit_max,
                }
            )
            if i % 10 == 0 or i == len(eval_examples):
                elapsed = time.time() - start_time
                rate = i / elapsed
                remaining = (len(eval_examples) - i) / rate if rate > 0 else 0
                print(
                    f"  [{i:3d}/{len(eval_examples)}] ({rate:.1f} ex/s, ~{remaining:.0f}s remaining)"
                )
    print(f"  done in {time.time() - start_time:.1f}s ({len(preds)} predictions)")
    return preds


def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("wrote", path)


# ── Baseline pass (adapter OFF) ──────────────────────────────────────────────
baseline_predictions = run_variant("BASELINE (untuned)", adapter_enabled=False)
write_jsonl(baseline_predictions, BASELINE_PRED_PATH)

# ── Fine-tuned pass (adapter ON) ─────────────────────────────────────────────
if has_adapter:
    finetuned_predictions = run_variant("FINE-TUNED (SFT+DPO)", adapter_enabled=True)
    write_jsonl(finetuned_predictions, FINETUNED_PRED_PATH)
else:
    finetuned_predictions = None
    print("\nNo adapter loaded — skipping the fine-tuned pass.")

## 6. Metrics

Computed for the column-description task, **for each variant**. The same `compute_metrics` routine scores the baseline and the fine-tuned predictions independently. (`compute_metrics` still emits an `overall` group; with a single task it equals `column_description`.)

Beyond ROUGE / BERTScore / length, this includes the **deterministic WA plain-language checks** from the capstone Output Evaluation Report (§9, implemented in §14.6): word count vs the ~50-word window (30–70), **sentence length** (mean words per sentence + share of sentences over the WA 20-word rule — the directly observed base-model failure), unexpanded acronyms, Flesch-Kincaid grade (WA target ≤ 9), the WA jargon substitution list, and "deadly 7 verbs" overuse. These are free, reproducible, and measure exactly the plain-language behavior the fine-tune claims to improve — which lexical-overlap metrics cannot see. The same checks are also run once on the **gold references** as a calibration anchor: with imperfect human golds, beating that calibration row on the rule checks is the real success criterion.

In [ ]:
import re as _re
from collections import defaultdict

# ── Deterministic WA plain-language checks ───────────────────────────────────
# Rule checks mirroring the capstone Output Evaluation Report (§9 / §14.6):
# free, reproducible, and they measure the plain-language behavior the
# fine-tune claims to improve — which ROUGE/BERTScore cannot see.
# WA word-substitution list — synced to the Evaluation Tool's
# backend/quality_checks.py _JARGON (the production deterministic checks), so
# "beat the golds on the rule checks" means the same thing in both repos.
# Like production, base forms only ("utilizes" is not flagged) — a shared
# limitation kept for cross-repo comparability.
WA_JARGON = [
    "utilize",
    "utilization",
    "prior to",
    "subsequent to",
    "in order to",
    "facilitate",
    "commence",
    "terminate",
    "endeavor",
    "furnish",
    "ascertain",
    "additional",
    "approximately",
    "in the event that",
    "with regard to",
    "in conjunction with",
    "pursuant to",
    "aforementioned",
    "henceforth",
    "leverage",
    "comprise",
    "demonstrate",
    "indicate",
    "methodology",
    "numerous",
    "obtain",
    "regarding",
    "sufficient",
]
DEADLY7 = {"am", "is", "are", "was", "were", "be", "been"}  # WA "deadly 7 verbs"
WORD_TARGET = (30, 70)  # column descriptions: ~50 words (WA soft-target window)
MAX_SENT_WORDS = 20  # WA rule: keep sentences under 20 words


def _sentences(text):
    # Decimal-safe split, synced to the Evaluation Tool's quality_checks.py:
    # punctuation ends a sentence only before whitespace/end-of-text, so "1.5"
    # cannot split into fake short sentences — which would deflate mean
    # words/sentence and under-count >20-word sentences, the headline
    # long-sentence metrics this eval exists to measure.
    return [s.strip() for s in _re.split(r"[.!?]+(?:\s+|$)", text) if s.strip()]


def _syllables(word):
    word = _re.sub(r"[^a-z]", "", word.lower())
    if not word:
        return 0
    n = len(_re.findall(r"[aeiouy]+", word))
    if word.endswith("e") and not word.endswith(("le", "ee")) and n > 1:
        n -= 1
    return max(1, n)


def fk_grade(text):
    """Flesch-Kincaid grade level (heuristic syllable counter); WA target <= 9."""
    sents, words = _sentences(text), _re.findall(r"[A-Za-z']+", text)
    if not sents or not words:
        return 0.0
    syllables = sum(_syllables(w) for w in words)
    return 0.39 * len(words) / len(sents) + 11.8 * syllables / len(words) - 15.59


_ACRONYM_RE = _re.compile(r"\b[A-Z][A-Z0-9]{1,}\b")


def _unexpanded_acronyms(text):
    """Port of the Evaluation Tool's quality_checks.py rule: an acronym counts
    as expanded when immediately followed by a parenthetical — "DOL (Department
    of Licensing)" — or introduced in parentheses — "Department of Licensing
    (DOL)". Same regex as production (digits allowed: "I5", "SR520")."""
    found, seen = [], set()
    for m in _ACRONYM_RE.finditer(text):
        token = m.group(0)
        if token in seen:
            continue
        seen.add(token)
        introduced_inline = text[m.end() : m.end() + 2].strip().startswith("(")
        if not introduced_inline and f"({token})" not in text:
            found.append(token)
    return found


def wa_checks(text):
    """Per-description rule checks; aggregated across a prediction set below."""
    words = _re.findall(r"[A-Za-z']+", text)
    sents = _sentences(text)
    low = text.lower()
    unexpanded = _unexpanded_acronyms(text)
    deadly_sents = sum(
        1
        for s in sents
        if set(w.lower() for w in _re.findall(r"[A-Za-z']+", s)) & DEADLY7
    )
    # Sentence length — the failure mode observed in base-model testing: the
    # LLM writes overlong sentences. Measured directly, per the WA <20-word rule.
    long_sents = sum(
        1 for s in sents if len(_re.findall(r"[A-Za-z']+", s)) > MAX_SENT_WORDS
    )
    return {
        "words": len(words),
        "in_word_target": WORD_TARGET[0] <= len(words) <= WORD_TARGET[1],
        "fk_grade": fk_grade(text),
        "sent_len_mean": (len(words) / len(sents)) if sents else 0.0,
        "share_long_sentences": (long_sents / len(sents)) if sents else 0.0,
        "has_unexpanded_acronym": bool(unexpanded),
        "has_jargon": any(_re.search(rf"\b{_re.escape(j)}\b", low) for j in WA_JARGON),
        "deadly7_overuse": bool(sents) and deadly_sents / len(sents) > 0.5,
    }


def aggregate_wa(texts):
    """Aggregate wa_checks over a list of descriptions -> table-ready metrics."""
    checks = [wa_checks(t) for t in texts]
    n = max(1, len(checks))
    return {
        "wa_words_mean": round(sum(c["words"] for c in checks) / n, 1),
        "pct_in_word_target": round(
            100 * sum(c["in_word_target"] for c in checks) / n, 1
        ),
        "fk_grade_mean": round(sum(c["fk_grade"] for c in checks) / n, 2),
        "wa_sent_len_mean": round(sum(c["sent_len_mean"] for c in checks) / n, 1),
        "pct_long_sentences": round(
            100 * sum(c["share_long_sentences"] for c in checks) / n, 1
        ),
        "pct_unexpanded_acronym": round(
            100 * sum(c["has_unexpanded_acronym"] for c in checks) / n, 1
        ),
        "pct_jargon": round(100 * sum(c["has_jargon"] for c in checks) / n, 1),
        "pct_deadly7_overuse": round(
            100 * sum(c["deadly7_overuse"] for c in checks) / n, 1
        ),
    }


def compute_metrics(predictions):
    """Score one variant's predictions: ROUGE, BERTScore, length/verbosity, WA checks.

    Returns {group: {metric: value}} for group in {overall, column_description}
    (a single task now, so overall == column_description).
    """
    by_task = defaultdict(list)
    for p in predictions:
        by_task[p["task"]].append(p)
    groups = {"overall": predictions, **by_task}
    results = {}

    # ---- ROUGE ----
    if DO_ROUGE:
        from rouge_score import rouge_scorer

        scorer = rouge_scorer.RougeScorer(
            ["rouge1", "rouge2", "rougeL"], use_stemmer=True
        )
        for name, rows in groups.items():
            agg = defaultdict(float)
            for p in rows:
                s = scorer.score(p["reference"], p["prediction"])
                for k, v in s.items():
                    agg[k] += v.fmeasure
            n = max(1, len(rows))
            results.setdefault(name, {}).update(
                {k: round(v / n, 4) for k, v in agg.items()}
            )

    # ---- BERTScore ----
    if DO_BERTSCORE:
        import bert_score

        for name, rows in groups.items():
            if name == "overall":
                continue  # scored per task; overall is the example-weighted mean below
            preds = [p["prediction"] for p in rows]
            refs = [p["reference"] for p in rows]
            _, _, f1 = bert_score.score(
                preds,
                refs,
                lang="en",
                model_type=BERTSCORE_MODEL,
                rescale_with_baseline=True,
            )
            results.setdefault(name, {})["bertscore_f1"] = round(float(f1.mean()), 4)
        # example-weighted overall
        tot = sum(results[t].get("bertscore_f1", 0) * len(by_task[t]) for t in by_task)
        results.setdefault("overall", {})["bertscore_f1"] = round(
            tot / max(1, len(predictions)), 4
        )

    # ---- Length / verbosity ratio ----
    if DO_LENGTH:

        def n_tok(s):
            return len(tokenizer.encode(s, add_special_tokens=False))

        for name, rows in groups.items():
            ratios = []
            for p in rows:
                r = n_tok(p["reference"])
                ratios.append(n_tok(p["prediction"]) / r if r else 0.0)
            n = max(1, len(ratios))
            results.setdefault(name, {}).update(
                {
                    "len_ratio_mean": round(sum(ratios) / n, 3),
                    "pct_over_threshold": round(
                        100 * sum(x > VERBOSITY_THRESHOLD for x in ratios) / n, 1
                    ),
                    "pct_hit_max_tokens": round(
                        100 * sum(bool(p.get("hit_max")) for p in rows) / n, 1
                    ),
                }
            )

    # ---- Deterministic WA plain-language checks ----
    if DO_WA_CHECKS:
        for name, rows in groups.items():
            results.setdefault(name, {}).update(
                aggregate_wa([p["prediction"] for p in rows])
            )
    return results


baseline_results = compute_metrics(baseline_predictions)
print("=== BASELINE (untuned) ===")
print(json.dumps(baseline_results, indent=2))

if finetuned_predictions is not None:
    finetuned_results = compute_metrics(finetuned_predictions)
    print("\n=== FINE-TUNED (SFT+DPO) ===")
    print(json.dumps(finetuned_results, indent=2))
else:
    finetuned_results = None

# Calibration anchor: what does sponsor-quality human text score on the same
# WA checks? (Gold references are not perfect plain language either — that is
# exactly why training curates targets and judges plain language against these
# deterministic rules instead of against raw imitation metrics.)
if DO_WA_CHECKS and baseline_predictions:
    print("\n=== WA checks on the GOLD references (calibration) ===")
    print(
        json.dumps(
            aggregate_wa([p["reference"] for p in baseline_predictions]), indent=2
        )
    )

## 7. Per-variant results tables + save

Saves each variant's metrics to its own JSON (`baseline_results.json`, `eval_results.json`) and renders its table.

In [ ]:
import pandas as pd

try:
    from IPython.display import display
except ImportError:  # plain-python fallback
    display = print


def save_results(results, path, adapter, n_examples):
    path.write_text(
        json.dumps(
            {
                "model_id": MODEL_ID,
                "adapter": adapter,
                "n_examples": n_examples,
                "verbosity_threshold": VERBOSITY_THRESHOLD,
                "metrics": results,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print("wrote", path)


# Baseline
save_results(baseline_results, BASELINE_RESULTS_PATH, None, len(baseline_predictions))
print("\nBASELINE (untuned):")
baseline_df = pd.DataFrame(baseline_results).T
baseline_df.index.name = "group"
display(baseline_df)

# Fine-tuned
if finetuned_results is not None:
    save_results(
        finetuned_results,
        FINETUNED_RESULTS_PATH,
        ADAPTER_PATH,
        len(finetuned_predictions),
    )
    print("\nFINE-TUNED (SFT+DPO):")
    finetuned_df = pd.DataFrame(finetuned_results).T
    finetuned_df.index.name = "group"
    display(finetuned_df)

## 8. Side-by-side metric comparison

Baseline vs. fine-tuned with the **delta** (`fine_tuned − baseline`) for the **column-description** task on the held-out test set.

For ROUGE and BERTScore, higher is better — a positive delta is an improvement. For `len_ratio_mean`, closer to **1.0** is better (≈ matching reference length); for `pct_over_threshold`, lower is better. The full breakdown (both variants) is saved to `comparison_results.json`.

In [ ]:
import pandas as pd
from collections import Counter

# Single task now, so the only meaningful group is column_description (it equals
# the "overall" group compute_metrics still emits).
_GROUP_ORDER = ["column_description"]
_GROUP_LABELS = {
    "column_description": "COLUMN_DESCRIPTION  (held-out test set)",
}
_METRIC_ORDER = [
    "rouge1",
    "rouge2",
    "rougeL",
    "bertscore_f1",
    "len_ratio_mean",
    "pct_over_threshold",
    "pct_hit_max_tokens",
    "wa_words_mean",
    "pct_in_word_target",
    "fk_grade_mean",
    "wa_sent_len_mean",
    "pct_long_sentences",
    "pct_unexpanded_acronym",
    "pct_jargon",
    "pct_deadly7_overuse",
]


def group_compare(group, baseline_results, finetuned_results):
    """baseline / fine_tuned / delta table for a single group (metric = index)."""
    b = baseline_results.get(group, {})
    f = finetuned_results.get(group, {})
    rows = []
    for m in _METRIC_ORDER:
        if m not in b and m not in f:
            continue
        bv, fv = b.get(m), f.get(m)
        delta = (
            round(fv - bv, 4)
            if isinstance(bv, (int, float)) and isinstance(fv, (int, float))
            else None
        )
        rows.append({"metric": m, "baseline": bv, "fine_tuned": fv, "delta": delta})
    return pd.DataFrame(rows).set_index("metric")


if finetuned_results is not None:
    # Persist the full breakdown (all groups, both variants).
    COMPARISON_PATH.write_text(
        json.dumps(
            {
                "model_id": MODEL_ID,
                "adapter": ADAPTER_PATH,
                "n_examples": len(baseline_predictions),
                "verbosity_threshold": VERBOSITY_THRESHOLD,
                "baseline": baseline_results,
                "fine_tuned": finetuned_results,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print("wrote", COMPARISON_PATH)

    task_counts = Counter(p["task"] for p in baseline_predictions)
    for g in _GROUP_ORDER:
        if g not in baseline_results and g not in finetuned_results:
            continue
        n = task_counts.get(g, len(baseline_predictions))
        print(f"\n===== {_GROUP_LABELS[g]}  (n={n}) =====")
        display(group_compare(g, baseline_results, finetuned_results))
else:
    print("No fine-tuned variant available — comparison skipped.")

## 9. Gold benchmark — sponsor "golden" datasets

A focused scoreboard on the sponsor's curated **golden** datasets only (tagged in `build_metadata_store.ipynb`, held out of training via the pinned test split). It re-uses the column-description predictions already generated in step 5 — **no extra generation** — and reports baseline vs. fine-tuned.

> The golden set is an *eval anchor*, not a training source. A few golden column descriptions include content that can't be inferred from the schema/stats alone (statute references, "best-effort" caveats), which lowers the achievable ROUGE/BERTScore on those rows — treat this as a directional sponsor-quality signal.

In [ ]:
# ── Gold benchmark: metrics restricted to the sponsor "golden" datasets ──
# Golden datasets (tagged in build_metadata_store.ipynb, pinned to the test split
# in finetune_descriptions.ipynb) are a sponsor-curated quality anchor. We re-score
# the SAME predictions from step 5, filtered to golden UIDs — no extra generation.
from collections import Counter as _Counter

golden_uids = {u for u, r in records.items() if r.get("golden")}


def _golden_subset(preds):
    return [p for p in preds if p["uid"] in golden_uids]


gold_base = _golden_subset(baseline_predictions)
print(
    f"Golden datasets in store: {len(golden_uids)} | gold benchmark examples: {len(gold_base)}"
)

if not gold_base:
    print(
        "\nNo golden datasets in the test split yet. Re-run build_metadata_store.ipynb "
        "with the FourMusketeers golden xlsx present (it tags records 'golden'), then "
        "rebuild splits.json in finetune_descriptions.ipynb."
    )
else:
    gold_counts = _Counter(p["task"] for p in gold_base)
    print(f"By task: {dict(gold_counts)}")
    gold_base_results = compute_metrics(gold_base)
    if finetuned_predictions is not None:
        gold_ft_results = compute_metrics(_golden_subset(finetuned_predictions))
        for g in _GROUP_ORDER:
            if g not in gold_base_results and g not in gold_ft_results:
                continue
            n = len(gold_base) if g == "overall" else gold_counts.get(g, 0)
            print(f"\n===== GOLD BENCHMARK · {_GROUP_LABELS[g]}  (n={n}) =====")
            display(group_compare(g, gold_base_results, gold_ft_results))
    else:
        print("\n=== GOLD BENCHMARK · BASELINE only (no adapter) ===")
        _df = pd.DataFrame(gold_base_results).T
        _df.index.name = "group"
        display(_df)

## 10. Novel-target subset — boilerplate-leakage control

Government portals reuse identical column descriptions across datasets ("County name.", "The year of the record."), so a held-out-by-dataset split does **not** guarantee the gold *text* is unseen: a model can score well on test rows by reciting boilerplate it memorized from train. `finetune_descriptions.ipynb` tags each test example whose normalized gold also appears among the train targets; this section re-scores the same step-5 predictions on the **novel** examples only. If the fine-tuned gains hold here, they are not an artifact of recited boilerplate.

In [ ]:
# ── Novel-target subset: drop test rows whose gold text also occurs in train ──
# Re-scores the SAME predictions from step 5 (predictions are index-aligned with
# eval_examples) — no extra generation.
novel_idx = [i for i, e in enumerate(eval_examples) if not e.get("target_in_train")]
print(f"novel-target examples: {len(novel_idx)}/{len(eval_examples)}")

if not novel_idx:
    print(
        "Every test target also appears among the train targets — the store is "
        "dominated by boilerplate descriptions; treat the headline metrics with care."
    )
elif finetuned_predictions is None:
    print("No fine-tuned predictions — novel-subset comparison skipped.")
else:
    novel_base = compute_metrics([baseline_predictions[i] for i in novel_idx])
    novel_ft = compute_metrics([finetuned_predictions[i] for i in novel_idx])
    print(
        f"\n===== NOVEL-TARGET SUBSET · COLUMN_DESCRIPTION  (n={len(novel_idx)}) ====="
    )
    display(group_compare("column_description", novel_base, novel_ft))

## 11. Improve-path slice — existing description present

Production's flagship flow sends an **existing description** in the REFERENCE block ("use it for context, do NOT copy its wording"); everything above measures the *undocumented column* path (empty reference). `finetune_descriptions.ipynb` renders an `improve_prompt_messages` variant for every test example whose gold could be degraded on one axis (jargonized / run-on / unexpanded / padded): the model sees the degraded text as the existing description and is scored against the same raw gold. Capped by `MAX_IMPROVE_EVAL` to bound GPU time. The adapter trained this path on a slice of SFT examples (plus the `improve_no_echo` DPO axis) — success looks like the fine-tuned variant *recovering* the clean description, with a lower **echo rate** (near-verbatim copying of the degraded reference) than the baseline.


In [ ]:
# ── Improve-path slice: the REFERENCE block carries a degraded gold ──────────
# Re-runs generation (both variants) on the improve_prompt_messages variants
# rendered by finetune_descriptions.ipynb §4: a degraded gold sits in the
# existing-description REFERENCE block and the model is scored against the same
# raw gold. The echo rate measures near-verbatim copying of the reference —
# column.md says "do NOT copy its wording", so lower is better.
import difflib

MAX_IMPROVE_EVAL = 60  # cap GPU time; None = every example with an improve prompt

improve_idx = [
    i for i, e in enumerate(eval_examples) if e.get("improve_prompt_messages")
]
if MAX_IMPROVE_EVAL:
    improve_idx = improve_idx[:MAX_IMPROVE_EVAL]
print(f"improve-path examples: {len(improve_idx)}")

if not improve_idx:
    print(
        "No improve_prompt_messages in test_examples.jsonl — re-run "
        "finetune_descriptions.ipynb §4 (June 2026 or later) to render them."
    )
elif finetuned_predictions is None:
    print("No adapter loaded — improve-path comparison skipped.")
else:

    def _run_improve(label, adapter_enabled):
        ctx = (
            contextlib.nullcontext()
            if (adapter_enabled or not has_adapter)
            else model.disable_adapter()
        )
        preds = []
        print(f"=== Generating improve-path: {label} ===")
        with ctx:
            for i in improve_idx:
                ex = eval_examples[i]
                pred, hit_max = generate(
                    ex["improve_prompt_messages"], MAX_NEW_TOKENS[ex["task"]]
                )
                preds.append(
                    {
                        "uid": ex["uid"],
                        "task": ex["task"],
                        "column": ex["column"],
                        "prediction": pred,
                        "reference": ex["target"],
                        "hit_max": hit_max,
                    }
                )
        return preds

    improve_base = _run_improve("BASELINE (untuned)", adapter_enabled=False)
    improve_ft = _run_improve("FINE-TUNED (SFT+DPO)", adapter_enabled=True)
    write_jsonl(improve_base, PROJECT_DIR / "improve_baseline_predictions.jsonl")
    write_jsonl(improve_ft, PROJECT_DIR / "improve_eval_predictions.jsonl")

    print(
        f"\n===== IMPROVE-PATH SLICE · COLUMN_DESCRIPTION  (n={len(improve_idx)}) ====="
    )
    display(
        group_compare(
            "column_description",
            compute_metrics(improve_base),
            compute_metrics(improve_ft),
        )
    )

    def _echo_rate(preds):
        """Share of predictions that are near-verbatim copies of the degraded
        reference shown in the prompt (similarity > 0.9)."""
        hits = 0
        for p, i in zip(preds, improve_idx):
            ref = eval_examples[i].get("improve_existing") or ""
            if (
                ref
                and difflib.SequenceMatcher(
                    None, p["prediction"].lower(), ref.lower()
                ).ratio()
                > 0.9
            ):
                hits += 1
        return 100 * hits / max(1, len(preds))

    print(
        f"\necho rate vs the degraded reference: "
        f"baseline {_echo_rate(improve_base):.0f}% | "
        f"fine-tuned {_echo_rate(improve_ft):.0f}%"
    )

## 12. Inspect predictions: base vs fine-tuned vs gold

Shows, side by side, the **untuned base model's** output, the **fine-tuned adapter's** output, and the **gold** reference for the *identical* prompt. Both prediction sets were already generated in step 5 (in the same order as `eval_examples`), so this just zips them together — no re-generation needed.

In [ ]:
# Side-by-side: base (untuned) vs fine-tuned vs gold, using the predictions
# already generated in step 5 (same order as eval_examples — no re-generation).

N_SHOW = 12  # column examples to show

if finetuned_predictions is not None:
    triples = list(zip(eval_examples, baseline_predictions, finetuned_predictions))[
        :N_SHOW
    ]
    print(f"\n########## column_description ({len(triples)} shown) ##########")
    for ex, bp, fp in triples:
        print(f"\n[{ex['uid']}/{ex['column']}]")
        print("BASE      :", bp["prediction"][:300])
        print("FINE-TUNED:", fp["prediction"][:300])
        print("GOLD      :", ex["target"][:300])
else:
    print("No adapter loaded — showing base (untuned) vs gold only.\n")
    pairs = list(zip(eval_examples, baseline_predictions))[:N_SHOW]
    print(f"\n########## column_description ({len(pairs)} shown) ##########")
    for ex, bp in pairs:
        print(f"\n[{ex['uid']}/{ex['column']}]")
        print("BASE:", bp["prediction"][:300])
        print("GOLD:", ex["target"][:300])